In [2]:
!pip install aiforge-framework


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastapi 0.115.6 requires starlette<0.42.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
gradio 6.5.1 requires starlette<1.0,>=0.40.0, but you have starlette 1.6.0 which is incompatible.
langchain-community 0.3.14 requires langchain<0.4.0,>=0.3.14, but you have langchain 1.4.2 which is incompatible.
langchain-community 0.3.14 requires langchain-core<0.4.0,>=0.3.29, but you have langchain-core 1.6.4 which is incompatible.
langchain-community 0.3.14 requires langsmith<0.3,>=0.1.125, but you have langsmith 0.14.0 which is incompatible.
langchain-text-splitters 0.3.4 requires langchain-core<0.4.0,>=0.3.26, but you have langchain-core 1.6.4 which is incompatible.
roboflow 1.2.13 requires idna==3.7, but you have idna 3.20 which is incompatible.

[notice] A new release of pip is available: 26.1.2 -> 26.2.1


  Using cached langchain_ollama-1.1.0-py3-none-any.whl.metadata (3.0 kB)
  Using cached langchain_protocol-0.0.19-py3-none-any.whl.metadata (2.4 kB)
  Using cached tiktoken-0.14.0-cp313-cp313-win_amd64.whl.metadata (6.8 kB)
  Using cached langgraph_checkpoint-4.2.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached langgraph_prebuilt-1.1.0-py3-none-any.whl.metadata (5.2 kB)
  Using cached ormsgpack-1.12.2-cp313-cp313-win_amd64.whl.metadata (3.3 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached pywin32-312-cp313-cp313-win_amd64.whl.metadata (11 kB)
  Using cached starlette-1.6.0-py3-none-any.whl.metadata (6.4 kB)
   ---------------------------------------- 0.0/572.0 kB ? eta -:--:--
   ---------------------------------------- 572.0/572.0 kB 9.8 MB/s  0:00:00
   ---------------------------------------- 0.0/1.2 MB ? eta -:--:--
   ---------------------------------------- 1.2/1.2 MB 17.9 MB/s  0:00:00
Using cached langchain_ollama-1.1.0-py3-none-any.whl (31 kB)

# utils

# Quiz Generation

In [3]:
from aipipe import LLM

In [4]:
from pathlib import Path
import os

PROJECT_ROOT = Path(
    r"C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe"
)

os.chdir(PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())

CONFIG_DIR = PROJECT_ROOT / "config"
LLM_CONFIG = CONFIG_DIR / "llm.yaml"

print("LLM config:", LLM_CONFIG)
print("LLM config exists:", LLM_CONFIG.exists())

Project root: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe
Working directory: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe
LLM config: C:\Users\user\Desktop\lms\DATAPRO-LMS\aipipe\config\llm.yaml
LLM config exists: True


In [5]:
from __future__ import annotations

import json
import os
from io import BytesIO
from pathlib import Path
from typing import Any, Literal

from dotenv import load_dotenv
from pydantic import BaseModel, Field, field_validator

load_dotenv(PROJECT_ROOT / ".env")

True

In [6]:
from aipipe import LLM
from aipipe.exceptions import SecurityError
from aipipe.security import SecurityManager

In [7]:
from quiz_ai import QuizAgent, QuizGenerator
from quiz_ai.security import (
    QuizSecurityPolicy,
    SecurityAgent,
)

In [8]:
Difficulty = Literal[
    "easy",
    "medium",
    "hard",
]

BloomLevel = Literal[
    "remember",
    "understand",
    "apply",
    "analyze",
    "evaluate",
    "create",
]

VALID_DIFFICULTIES = {
    "easy",
    "medium",
    "hard",
}

VALID_BLOOM_LEVELS = {
    "remember",
    "understand",
    "apply",
    "analyze",
    "evaluate",
    "create",
}

DEFAULT_COUNT = 10

DEFAULT_DIFFICULTY = {
    "easy": 3,
    "medium": 4,
    "hard": 3,
}

DEFAULT_BLOOM = {
    "remember": 2,
    "understand": 2,
    "apply": 2,
    "analyze": 2,
    "evaluate": 1,
    "create": 1,
}

In [9]:
class QuizGenerationRequest(BaseModel):

    content: str = Field(
        ...,
        min_length=1,
        description="Learning content.",
    )

    count: int = Field(
        default=5,
        ge=1,
        le=100,
        description="Number of questions.",
    )

    difficulty: dict[Difficulty, int] | None = None

    bloom: dict[BloomLevel, int] | None = None

    @field_validator("content")
    @classmethod
    def validate_content(cls, value: str) -> str:

        value = value.strip()

        if not value:
            raise ValueError("content cannot be empty.")

        return value

    @field_validator("difficulty")
    @classmethod
    def validate_difficulty(
        cls,
        value: dict[Difficulty, int] | None,
    ):
        if value is None:
            return None

        for key, amount in value.items():

            if not isinstance(amount, int):
                raise ValueError(
                    f"Difficulty '{key}' must be an integer."
                )

            if amount < 0:
                raise ValueError(
                    f"Difficulty '{key}' cannot be negative."
                )

        return value

    @field_validator("bloom")
    @classmethod
    def validate_bloom(
        cls,
        value: dict[BloomLevel, int] | None,
    ):
        if value is None:
            return None

        for key, amount in value.items():

            if not isinstance(amount, int):
                raise ValueError(
                    f"Bloom level '{key}' must be an integer."
                )

            if amount < 0:
                raise ValueError(
                    f"Bloom level '{key}' cannot be negative."
                )

        return value

In [10]:
def resolve_distributions(
    count: int,
    difficulty: dict[str, int] | None = None,
    bloom: dict[str, int] | None = None,
):
    if difficulty is None:

        if count == DEFAULT_COUNT:
            difficulty = DEFAULT_DIFFICULTY.copy()
        else:
            difficulty = {
                "medium": count
            }

    if bloom is None:

        if count == DEFAULT_COUNT:
            bloom = DEFAULT_BLOOM.copy()
        else:
            bloom = {
                "understand": count
            }

    return difficulty, bloom

In [11]:
def validate_generation_request(
    content: str,
    count: int,
    difficulty: dict[str, int],
    bloom: dict[str, int],
) -> None:

    if not isinstance(content, str):
        raise TypeError("content must be a string.")

    content = content.strip()

    if not content:
        raise ValueError("content cannot be empty.")

    if count < 1:
        raise ValueError("count must be greater than 0.")

    if count > 100:
        raise ValueError(
            "count cannot be greater than 100."
        )

    unknown_difficulty = (
        set(difficulty) - VALID_DIFFICULTIES
    )

    if unknown_difficulty:
        raise ValueError(
            f"Invalid difficulty values: "
            f"{sorted(unknown_difficulty)}"
        )

    unknown_bloom = (
        set(bloom) - VALID_BLOOM_LEVELS
    )

    if unknown_bloom:
        raise ValueError(
            f"Invalid Bloom levels: "
            f"{sorted(unknown_bloom)}"
        )

    if any(
        not isinstance(value, int)
        or value < 0
        for value in difficulty.values()
    ):
        raise ValueError(
            "Difficulty counts must be non-negative integers."
        )

    if any(
        not isinstance(value, int)
        or value < 0
        for value in bloom.values()
    ):
        raise ValueError(
            "Bloom counts must be non-negative integers."
        )

    if sum(difficulty.values()) != count:
        raise ValueError(
            "Difficulty distribution must equal question count. "
            f"Expected {count}, "
            f"got {sum(difficulty.values())}."
        )

    if sum(bloom.values()) != count:
        raise ValueError(
            "Bloom distribution must equal question count. "
            f"Expected {count}, "
            f"got {sum(bloom.values())}."
        )

In [12]:
def create_security_agent() -> SecurityAgent:

    security_manager = SecurityManager()

    policy = QuizSecurityPolicy(
        enabled=True,
        block_jailbreak=True,
        block_document_injection=True,
        block_quiz_manipulation=True,
        block_tool_abuse=True,
        block_role_override=True,
        block_system_prompt_extraction=True,
        block_secret_extraction=True,
        fail_closed=True,
    )

    return SecurityAgent(
        security_manager=security_manager,
        policy=policy,
    )

In [13]:
def generate_secure_quiz(
    content: str,
    count: int,
    difficulty: dict[str, int],
    bloom: dict[str, int],
) -> dict[str, Any]:

    # 1. Validate request
    validate_generation_request(
        content=content,
        count=count,
        difficulty=difficulty,
        bloom=bloom,
    )

    print("=" * 80)
    print("SECURE QUIZ GENERATION")
    print("=" * 80)

    print(f"Characters : {len(content)}")
    print(f"Words      : {len(content.split())}")
    print(f"Questions  : {count}")
    print(f"Difficulty : {difficulty}")
    print(f"Bloom      : {bloom}")

    # 2. Security
    print("\n1. SECURITY INITIALIZATION")

    security_agent = create_security_agent()

    print("SecurityAgent : READY")

    # 3. Check learning content
    print("\n2. LEARNING CONTENT SECURITY CHECK")

    security_result = (
        security_agent.check_learning_content(
            content
        )
    )

    print("Allowed    :", security_result.allowed)
    print("Risk       :", security_result.risk)
    print("Categories :", security_result.categories)
    print("Message    :", security_result.message)

    if not security_result.allowed:
        raise SecurityError(
            security_result.message
            or "Learning content blocked by security policy.",
            result=security_result,
        )

    # 4. Sanitize
    print("\n3. CONTENT SANITIZATION")

    safe_content = (
        security_agent.secure_learning_content(
            content
        )
    )

    print(
        "Sanitized characters :",
        len(safe_content),
    )

    # 5. Load LLM
    print("\n4. LOAD LLM")

    if not LLM_CONFIG.exists():
        raise FileNotFoundError(
            f"LLM configuration not found: {LLM_CONFIG}"
        )

    llm = LLM.from_config(
        str(LLM_CONFIG)
    )

    print("LLM:", llm.info())

    # 6. Quiz Agent
    print("\n5. CREATE QUIZ AGENT")

    agent = QuizAgent(
        llm=llm,
        tools=None,
        memory=None,
        security=security_agent,
        name="secure_quiz_agent",
    )

    print("QuizAgent : READY")

    # 7. Quiz Generator
    print("\n6. CREATE QUIZ GENERATOR")

    generator = QuizGenerator(
        agent=agent,
        batch_size=min(10, count),
        max_retries=2,
    )

    print("QuizGenerator : READY")

    # 8. Generate
    print("\n7. GENERATING QUIZ")

    try:

        generated_quiz = generator.generate(
            content=safe_content,
            count=count,
            difficulty=difficulty,
            bloom=bloom,
        )

    except SecurityError:
        raise

    except Exception as exc:
        raise RuntimeError(
            f"Quiz generation failed: {exc}"
        ) from exc

    print("LLM generation : COMPLETE")

    return generated_quiz

In [14]:
content = """
Python is a high-level programming language.
It is widely used for web development, data science,
machine learning, automation, and scripting.

Python uses variables to store data.
Common data types include strings, integers,
floating-point numbers, lists, tuples, dictionaries,
and sets.

Python supports conditional statements such as if,
elif, and else. It also supports loops such as
for and while.
"""

count = 5

difficulty = {
    "easy": 2,
    "medium": 2,
    "hard": 1,
}

bloom = {
    "remember": 1,
    "understand": 1,
    "apply": 1,
    "analyze": 1,
    "evaluate": 1,
    "create": 0,
}

result = generate_secure_quiz(
    content=content,
    count=count,
    difficulty=difficulty,
    bloom=bloom,
)

SECURE QUIZ GENERATION
Characters : 404
Words      : 58
Questions  : 5
Difficulty : {'easy': 2, 'medium': 2, 'hard': 1}
Bloom      : {'remember': 1, 'understand': 1, 'apply': 1, 'analyze': 1, 'evaluate': 1, 'create': 0}

1. SECURITY INITIALIZATION
SecurityAgent : READY

2. LEARNING CONTENT SECURITY CHECK
Allowed    : True
Risk       : none
Categories : []
Message    : Security check passed.

3. CONTENT SANITIZATION
Sanitized characters : 404

4. LOAD LLM
LLM: {'provider': 'ollama', 'model': 'gemma4:31b', 'api_key': None, 'api_key_env': 'OLLAMA_API_KEY', 'base_url': 'https://ollama.com', 'temperature': 0.0, 'max_tokens': 4096, 'generation_type': 'text', 'supports_streaming': True, 'supports_tools': True, 'supports_structured_output': True, 'timeout': 120.0, 'connection_mode': 'ollama_cloud', 'client': {'class': 'LLM', 'model_class': 'OllamaCloudAPIModel'}}

5. CREATE QUIZ AGENT
QuizAgent : READY

6. CREATE QUIZ GENERATOR
QuizGenerator : READY

7. GENERATING QUIZ
SECURE QUIZ BATCH GENERA

# personalized learning

Set project root


In [2]:
# ============================================================
# CELL 1 - Imports and Project Configuration
# ============================================================

import os
import sys
import json
import re
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

In [4]:
# ============================================================
# CELL 2 - Project Paths
# ============================================================

import os
import sys
from pathlib import Path

# Dynamically get the current directory
PROJECT_ROOT = Path.cwd().resolve()

# Add project root to Python path
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project Root:", PROJECT_ROOT)
print("Current Directory:", Path.cwd())

Project Root: C:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\personalized_ai\research
Current Directory: c:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\personalized_ai\research


In [5]:
db_config = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "root",
    "password": "Mysql@2025",
    "database": "datapro_lms",
    "connection_timeout": 10
}

In [6]:
import mysql.connector
from mysql.connector import Error

def get_quiz_question(db_config, quiz_id, question_number):
    """
    Fetch a specific quiz question from tbl_quiz_questions.

    Args:
        db_config (dict): MySQL database configuration.
        quiz_id (int): Quiz ID.
        question_number (int): Question number.

    Returns:
        dict | None: Dictionary containing question_text, options_json,
                     correct_answer, difficulty and bloom_level.
                     Returns None if no matching question is found.
    """
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT
                question_text,
                options_json,
                correct_answer,
                difficulty,
                bloom_level
            FROM datapro_lms.tbl_quiz_questions
            WHERE quiz_id = %s
              AND question_number = %s
            LIMIT 1
        """

        with connection.cursor(dictionary=True) as cursor:
            cursor.execute(query, (quiz_id, question_number))
            result = cursor.fetchone()

        return result

    except Error as e:
        print(f"Database error: {e}")
        raise

    finally:
        if connection and connection.is_connected():
            connection.close()

In [9]:
result = get_quiz_question(
    db_config=db_config,
    quiz_id=13,
    question_number=3
)

if result:
    print("Question:", result["question_text"])
    print("Options:", result["options_json"])
    print("Correct Answer:", result["correct_answer"])
    print("Difficulty:", result["difficulty"])
    print("Bloom Level:", result["bloom_level"])
else:
    print("Question not found.")

Question: In CSS, what is the main difference between margin and padding?
Options: ["Margin is inside the border, padding is outside", "Margin is for text color, padding is for background color", "Margin is the space outside the border, padding is the space inside the border", "There is no difference between margin and padding"]
Correct Answer: Margin is the space outside the border, padding is the space inside the border
Difficulty: medium
Bloom Level: understand


In [10]:
import mysql.connector
from mysql.connector import Error

def get_wrong_answers(db_config, student_id, quiz_id):
    """
    Retrieve all wrong answers for a student in a particular quiz.

    Args:
        db_config (dict): MySQL database configuration.
        student_id (str): Student ID.
        quiz_id (int): Quiz ID.

    Returns:
        list[dict]: List of wrong answers with question details
                    and selected answers.
    """
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT
                qa.attempt_id,
                a.question_id,
                q.question_number,
                q.question_text,
                q.options_json,
                a.selected_answer,
                q.correct_answer,
                q.difficulty,
                q.bloom_level
            FROM datapro_lms.tbl_quiz_attempts AS qa
            INNER JOIN datapro_lms.tbl_quiz_attempt_answers AS a
                ON qa.attempt_id = a.attempt_id
            INNER JOIN datapro_lms.tbl_quiz_questions AS q
                ON a.question_id = q.question_id
            WHERE qa.student_id = %s
              AND qa.quiz_id = %s
              AND a.is_correct = 0
            ORDER BY qa.submitted_at DESC, q.question_number ASC
        """

        with connection.cursor(dictionary=True) as cursor:
            cursor.execute(query, (student_id, quiz_id))
            results = cursor.fetchall()

        return results

    except Error as e:
        print(f"Database error: {e}")
        raise

    finally:
        if connection and connection.is_connected():
            connection.close()

In [11]:
wrong_answers = get_wrong_answers(
    db_config=db_config,
    student_id="ITDA1267",
    quiz_id=1
)

for item in wrong_answers:
    print(f"\nQuestion Number: {item['question_number']}")
    print(f"Question: {item['question_text']}")
    print(f"Selected Answer: {item['selected_answer']}")
    print(f"Correct Answer: {item['correct_answer']}")
    print(f"Difficulty: {item['difficulty']}")
    print(f"Bloom Level: {item['bloom_level']}")


Question Number: 1
Question: Which of the following best describes the purpose of 'syntax' in Python?
Selected Answer: A tool used for debugging variable errors
Correct Answer: The set of rules that defines how a Python program is written and structured
Difficulty: None
Bloom Level: None

Question Number: 2
Question: What is the primary function of a variable in Python?
Selected Answer: To create permanent comments in the code
Correct Answer: To store data values for later use in the program
Difficulty: None
Bloom Level: None

Question Number: 3
Question: In the context of Python, what is an 'identifier'?
Selected Answer: A special comment used for documentation
Correct Answer: The name given to entities like variables, functions, or classes
Difficulty: None
Bloom Level: None

Question Number: 4
Question: What is the main purpose of using comments in a Python script?
Selected Answer: To define the variables used in the program
Correct Answer: To provide explanations and make the code 

In [14]:
import mysql.connector

def get_module_ids(db_config, course_id):
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT module_id
            FROM tbl_course_modules
            WHERE course_id = %s
        """

        with connection.cursor() as cursor:
            cursor.execute(query, (course_id,))
            rows = cursor.fetchall()

        return [row[0] for row in rows]

    finally:
        if connection and connection.is_connected():
            connection.close()

In [21]:
module_ids = get_module_ids(
    db_config=db_config,
    course_id=1
)

print(module_ids)

[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]


In [22]:
import mysql.connector

def get_module_id(db_config, course_id, module_title):
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT module_id
            FROM tbl_course_modules
            WHERE course_id = %s
              AND module_title = %s
            LIMIT 1
        """

        with connection.cursor() as cursor:
            cursor.execute(query, (course_id, module_title))
            result = cursor.fetchone()

        return result[0] if result else None

    finally:
        if connection and connection.is_connected():
            connection.close()

In [40]:
module_id = get_module_id(
    db_config=db_config,
    course_id=1,
    module_title="Onboarding &  Computer  Fundamentals"
)

print(module_id)

3


# Final

In [47]:
import mysql.connector
from mysql.connector import Error

def get_module_wrong_answers(db_config, course_id, module_id, student_id):
    """
    Check quiz attendance and retrieve wrong answers for each quiz
    associated with a module.

    Returns:
        dict: Attendance status and wrong answers for each quiz.
    """

    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        query = """
            SELECT
                cc.quiz_id,
                MIN(cc.sequence_no) AS sequence_no,
                COUNT(DISTINCT qa.attempt_id) AS attempt_count
            FROM tbl_course_content AS cc
            LEFT JOIN tbl_quiz_attempts AS qa
                ON cc.quiz_id = qa.quiz_id
                AND cc.course_id = qa.course_id
                AND qa.student_id = %s
            WHERE cc.course_id = %s
              AND cc.module_id = %s
              AND cc.quiz_id IS NOT NULL
              AND (cc.is_deleted = 0 OR cc.is_deleted IS NULL)
            GROUP BY cc.quiz_id
            ORDER BY sequence_no ASC
        """

        with connection.cursor(dictionary=True) as cursor:
            cursor.execute(query, (student_id, course_id, module_id))
            quiz_rows = cursor.fetchall()

    except Error as e:
        print(f"Database error: {e}")
        raise

    finally:
        if connection and connection.is_connected():
            connection.close()

    results = []

    # Check attendance and retrieve wrong answers
    for row in quiz_rows:
        quiz_id = row["quiz_id"]
        attempt_count = row["attempt_count"]

        if attempt_count == 0:
            results.append({
                "quiz_id": quiz_id,
                "status": "Not attended",
                "attempt_count": 0,
                "wrong_answers": []
            })
            continue

        wrong_answers = get_wrong_answers(
            db_config=db_config,
            student_id=student_id,
            quiz_id=quiz_id
        )

        results.append({
            "quiz_id": quiz_id,
            "status": "Attended",
            "attempt_count": attempt_count,
            "wrong_answers": wrong_answers
        })

    return results

In [55]:
results = get_module_wrong_answers(
    db_config=db_config,
    course_id=1,
    module_id=8,
    student_id="ITDA1282"
)

for item in results:
    print(f"\nQuiz ID: {item['quiz_id']}")
    print(f"Status: {item['status']}")
    print(f"Attempts: {item['attempt_count']}")

    if item["status"] == "Not attended":
        print("Student has not attended this quiz.")
    elif item["wrong_answers"]:
        for answer in item["wrong_answers"]:
            print("\nQuestion:", answer["question_text"])
            print("Selected answer:", answer["selected_answer"])
            print("Correct answer:", answer["correct_answer"])
            print("Difficulty:", answer["difficulty"])
            print("Bloom level:", answer["bloom_level"])
    else:
        print("No wrong answers.")


Quiz ID: 9
Status: Not attended
Attempts: 0
Student has not attended this quiz.

Quiz ID: 4
Status: Attended
Attempts: 1

Question: What distinguishes CSS Grid from Flexbox according to the course summary?
Selected answer: Grid is primarily for one-dimensional layouts, while Flexbox is for two-dimensional
Correct answer: Grid is mainly used for two-dimensional layouts, while Flexbox is primarily for one-dimensional layouts
Difficulty: None
Bloom level: None

Question: What is the primary difference between fixed-size layouts and flexible layouts in CSS Grid?
Selected answer: Fixed layouts are two-dimensional, while flexible layouts are one-dimensional
Correct answer: Fixed layouts use pixels, while flexible layouts adapt to available screen space
Difficulty: None
Bloom level: None

Question: Which property is used to control the rows of a grid?
Selected answer: display: rows;
Correct answer: grid-template-rows
Difficulty: None
Bloom level: None


In [57]:
results

[{'quiz_id': 9,
  'status': 'Not attended',
  'attempt_count': 0,
  'wrong_answers': []},
 {'quiz_id': 4,
  'status': 'Attended',
  'attempt_count': 1,
  'wrong_answers': [{'attempt_id': 45,
    'question_id': 31,
    'question_number': 1,
    'question_text': 'What distinguishes CSS Grid from Flexbox according to the course summary?',
    'options_json': '["Flexbox is two-dimensional, while Grid is one-dimensional", "Grid is primarily for one-dimensional layouts, while Flexbox is for two-dimensional", "Grid is mainly used for two-dimensional layouts, while Flexbox is primarily for one-dimensional layouts", "Grid cannot be used together with Flexbox"]',
    'selected_answer': 'Grid is primarily for one-dimensional layouts, while Flexbox is for two-dimensional',
    'correct_answer': 'Grid is mainly used for two-dimensional layouts, while Flexbox is primarily for one-dimensional layouts',
    'difficulty': None,
    'bloom_level': None},
   {'attempt_id': 45,
    'question_id': 36,
    

In [59]:
from pathlib import Path
from dotenv import load_dotenv

from aipipe import LLM

# Dynamically get the project root
PROJECT_ROOT = Path.cwd()

# If running from a subdirectory, locate the project root
while not (PROJECT_ROOT / "config" / "llm.yaml").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("AIPipe project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load .env
load_dotenv(PROJECT_ROOT / ".env")

# LLM configuration
CONFIG_DIR = PROJECT_ROOT / "config"
LLM_CONFIG = CONFIG_DIR / "llm.yaml"

print("Project Root:", PROJECT_ROOT)
print("Config:", LLM_CONFIG)
print("Config exists:", LLM_CONFIG.exists())

# Load LLM
llm = LLM.from_config(str(LLM_CONFIG))

Project Root: c:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe
Config: c:\Users\Deepak\LMS Datapro\DATAPRO-LMS\aipipe\config\llm.yaml
Config exists: True


In [60]:
llm.generate("hello world")

LLMResponse(content='Hello World! How can I help you today?', metadata={'provider': 'ollama', 'model': 'gemma4:31b'})

In [70]:
def summarize_student_module_performance(
    db_config,
    course_id,
    module_id,
    student_id,
    llm,
    batch_size=5,
    max_retries=2
):
    """
    Generate a detailed student performance report for a module.

    Includes:
      - Module quiz attendance and scores
      - Incorrect answers and correct answers
      - Difficulty and Bloom-level analysis
      - Detailed knowledge gaps
      - Possible misconceptions and explanations
      - Topics to revisit and practice recommendations
      - Personalized revision roadmap

    Uses the supplied AIPipe LLM and returns a Python dictionary.
    """

    import json
    import re
    from decimal import Decimal
    from datetime import date, datetime
    import mysql.connector
    from mysql.connector import Error

    # ---------------------------------------------------------
    # 1. Helper: Convert database values to JSON-safe values
    # ---------------------------------------------------------
    def json_safe(value):
        if isinstance(value, Decimal):
            return float(value)
        if isinstance(value, (datetime, date)):
            return value.isoformat()
        if isinstance(value, bytes):
            return value.decode("utf-8", errors="replace")
        return value

    # ---------------------------------------------------------
    # 2. Helper: Extract text from an LLM response
    # ---------------------------------------------------------
    def get_response_text(response):
        if isinstance(response, str):
            return response

        content = getattr(response, "content", None)

        if isinstance(content, str):
            return content

        if isinstance(content, list):
            parts = []
            for item in content:
                if isinstance(item, str):
                    parts.append(item)
                elif isinstance(item, dict) and item.get("text"):
                    parts.append(item["text"])
            return "\n".join(parts)

        return str(response)

    # ---------------------------------------------------------
    # 3. Helper: Parse JSON, including JSON in code fences
    # ---------------------------------------------------------
    def parse_json_response(response):
        text = get_response_text(response).strip()

        text = re.sub(
            r"^\s*```(?:json)?\s*|\s*```\s*$",
            "",
            text,
            flags=re.IGNORECASE
        ).strip()

        decoder = json.JSONDecoder()

        for index, char in enumerate(text):
            if char not in "{[":
                continue

            try:
                result, _ = decoder.raw_decode(text[index:])
                if isinstance(result, (dict, list)):
                    return result
            except json.JSONDecodeError:
                continue

        raise ValueError(
            "LLM returned incomplete or invalid JSON. "
            f"Response length: {len(text)} characters. "
            f"Response ending: {text[-500:]}"
        )

    # ---------------------------------------------------------
    # 4. Helper: Call LLM with retries
    # ---------------------------------------------------------
    def invoke_json(prompt, retries=None):
        retries = max_retries if retries is None else retries
        last_error = None

        for attempt in range(retries + 1):
            current_prompt = prompt

            if attempt > 0:
                current_prompt += """

Your previous response could not be parsed as complete JSON.
Generate a shorter response.
Return exactly one complete JSON object.
Do not use Markdown fences or additional text.
Ensure that every opening bracket and quotation mark is closed.
"""

            response = llm.invoke(current_prompt)
            raw_text = get_response_text(response)

            try:
                return parse_json_response(raw_text)
            except (ValueError, json.JSONDecodeError) as exc:
                last_error = exc
                print(
                    f"LLM JSON attempt {attempt + 1}/{retries + 1} failed. "
                    f"Response length: {len(raw_text)} characters."
                )

        raise ValueError(
            f"Unable to obtain valid JSON after {retries + 1} attempts: "
            f"{last_error}"
        )

    # ---------------------------------------------------------
    # 5. Fetch module quizzes and student attempts
    # ---------------------------------------------------------
    connection = None

    try:
        connection = mysql.connector.connect(**db_config)

        with connection.cursor(dictionary=True) as cursor:

            # Get quizzes belonging to the specified module.
            cursor.execute(
                """
                SELECT
                    quiz_id,
                    MIN(sequence_no) AS sequence_no
                FROM datapro_lms.tbl_course_content
                WHERE course_id = %s
                  AND module_id = %s
                  AND quiz_id IS NOT NULL
                  AND (is_deleted = 0 OR is_deleted IS NULL)
                GROUP BY quiz_id
                ORDER BY MIN(sequence_no), quiz_id
                """,
                (course_id, module_id)
            )

            quiz_rows = cursor.fetchall()
            quiz_ids = [row["quiz_id"] for row in quiz_rows]

            base_report = {
                "course_id": course_id,
                "module_id": module_id,
                "student_id": student_id,
                "total_quizzes": len(quiz_ids),
                "attempted_quizzes": 0,
                "unattempted_quizzes": [],
                "total_attempts": 0,
                "total_questions": 0,
                "total_correct": 0,
                "total_incorrect": 0,
                "average_percentage": 0.0,
                "quiz_results": [],
                "difficulty_analysis": {},
                "bloom_analysis": {},
                "knowledge_gaps": [],
                "overall_summary": "",
                "strengths": [],
                "recommendations": [],
                "revision_roadmap": [],
                "data_limitations": []
            }

            if not quiz_ids:
                base_report["overall_summary"] = (
                    "No quizzes were found for the specified module."
                )
                return base_report

            placeholders = ", ".join(["%s"] * len(quiz_ids))

            cursor.execute(
                f"""
                SELECT
                    attempt_id,
                    quiz_id,
                    score,
                    total_questions,
                    correct_count,
                    percent,
                    status,
                    submitted_at,
                    duration_seconds
                FROM datapro_lms.tbl_quiz_attempts
                WHERE student_id = %s
                  AND course_id = %s
                  AND quiz_id IN ({placeholders})
                ORDER BY submitted_at ASC, attempt_id ASC
                """,
                (student_id, course_id, *quiz_ids)
            )

            attempts = cursor.fetchall()

            # Fetch wrong answers for the exact attempts retrieved.
            attempt_ids = [row["attempt_id"] for row in attempts]
            wrong_answers = []

            if attempt_ids:
                attempt_placeholders = ", ".join(
                    ["%s"] * len(attempt_ids)
                )

                cursor.execute(
                    f"""
                    SELECT
                        qa.attempt_id,
                        qa.quiz_id,
                        a.question_id,
                        q.question_number,
                        q.question_text,
                        q.options_json,
                        a.selected_answer,
                        q.correct_answer,
                        q.difficulty,
                        q.bloom_level
                    FROM datapro_lms.tbl_quiz_attempts AS qa
                    INNER JOIN datapro_lms.tbl_quiz_attempt_answers AS a
                        ON qa.attempt_id = a.attempt_id
                    INNER JOIN datapro_lms.tbl_quiz_questions AS q
                        ON a.question_id = q.question_id
                    WHERE qa.attempt_id IN ({attempt_placeholders})
                      AND a.is_correct = 0
                    ORDER BY qa.submitted_at ASC, q.question_number ASC
                    """,
                    tuple(attempt_ids)
                )

                wrong_answers = cursor.fetchall()

    except Error as exc:
        raise RuntimeError(
            f"Database error while retrieving student performance: {exc}"
        ) from exc

    finally:
        if connection and connection.is_connected():
            connection.close()

    # ---------------------------------------------------------
    # 6. Build quiz results and calculate statistics
    # ---------------------------------------------------------
    attempts_by_quiz = {}
    for attempt in attempts:
        attempts_by_quiz.setdefault(attempt["quiz_id"], []).append(attempt)

    wrong_by_attempt = {}
    for answer in wrong_answers:
        wrong_by_attempt.setdefault(
            answer["attempt_id"], []
        ).append(answer)

    all_wrong_answers = []
    total_questions = 0
    total_correct = 0
    percentages = []
    difficulty_stats = {}
    bloom_stats = {}

    quiz_results = []

    for quiz_id in quiz_ids:
        quiz_attempts = attempts_by_quiz.get(quiz_id, [])

        if not quiz_attempts:
            base_report["unattempted_quizzes"].append(quiz_id)
            quiz_results.append({
                "quiz_id": quiz_id,
                "attended": False,
                "attempt_count": 0,
                "attempts": [],
                "message": "The student has not attempted this quiz."
            })
            continue

        attempt_results = []

        for attempt in quiz_attempts:
            attempt_id = attempt["attempt_id"]
            attempt_wrong = wrong_by_attempt.get(attempt_id, [])

            total = int(attempt.get("total_questions") or 0)
            correct = int(attempt.get("correct_count") or 0)
            incorrect = max(0, total - correct)

            total_questions += total
            total_correct += correct

            if attempt.get("percent") is not None:
                percentages.append(float(attempt["percent"]))

            for answer in attempt_wrong:
                all_wrong_answers.append(answer)

                difficulty = (
                    str(answer.get("difficulty") or "Unknown").strip()
                )
                bloom = (
                    str(answer.get("bloom_level") or "Unknown").strip()
                )

                difficulty_stats.setdefault(
                    difficulty,
                    {"incorrect": 0}
                )
                difficulty_stats[difficulty]["incorrect"] += 1

                bloom_stats.setdefault(
                    bloom,
                    {"incorrect": 0}
                )
                bloom_stats[bloom]["incorrect"] += 1

            attempt_results.append({
                "attempt_id": attempt_id,
                "score": json_safe(attempt.get("score")),
                "total_questions": total,
                "correct_count": correct,
                "incorrect_count": incorrect,
                "percentage": json_safe(attempt.get("percent")),
                "status": attempt.get("status"),
                "submitted_at": json_safe(attempt.get("submitted_at")),
                "duration_seconds": attempt.get("duration_seconds"),
                "wrong_answers": attempt_wrong
            })

        quiz_results.append({
            "quiz_id": quiz_id,
            "attended": True,
            "attempt_count": len(quiz_attempts),
            "attempts": attempt_results
        })

    base_report["attempted_quizzes"] = sum(
        1 for quiz in quiz_results if quiz["attended"]
    )
    base_report["total_attempts"] = len(attempts)
    base_report["total_questions"] = total_questions
    base_report["total_correct"] = total_correct
    base_report["total_incorrect"] = max(
        0, total_questions - total_correct
    )
    base_report["average_percentage"] = (
        round(sum(percentages) / len(percentages), 2)
        if percentages else 0.0
    )
    base_report["quiz_results"] = quiz_results
    base_report["difficulty_analysis"] = difficulty_stats
    base_report["bloom_analysis"] = bloom_stats

    # Convert DB values before serializing the report.
    def make_serializable(obj):
        if isinstance(obj, dict):
            return {
                str(key): make_serializable(value)
                for key, value in obj.items()
            }
        if isinstance(obj, list):
            return [make_serializable(item) for item in obj]
        return json_safe(obj)

    base_report = make_serializable(base_report)
    all_wrong_answers = make_serializable(all_wrong_answers)

    # ---------------------------------------------------------
    # 7. Handle students with no attempts
    # ---------------------------------------------------------
    if not attempts:
        base_report["overall_summary"] = (
            "The student has not attempted any quizzes in this module. "
            "There is insufficient assessment evidence to identify "
            "knowledge gaps or evaluate learning performance."
        )
        base_report["data_limitations"] = [
            "No quiz attempts were recorded for this module.",
            "Knowledge gaps and strengths cannot be established "
            "from the available assessment data."
        ]
        base_report["recommendations"] = [
            "Complete the module quizzes to establish a performance baseline."
        ]
        return base_report

    # ---------------------------------------------------------
    # 8. Generate detailed knowledge gaps in small batches
    # ---------------------------------------------------------
    knowledge_gaps = []

    if all_wrong_answers:
        for start in range(0, len(all_wrong_answers), batch_size):
            batch = all_wrong_answers[start:start + batch_size]

            prompt = f"""
You are an educational assessment analyst.

Analyze the following incorrectly answered quiz questions.
Produce a detailed but concise JSON object with this exact structure:

{{
  "knowledge_gaps": [
    {{
      "question_id": 0,
      "question_number": 0,
      "concept": "Concept name",
      "topic": "Main topic",
      "subtopic": "Specific subtopic",
      "priority": "high",
      "evidence_strength": "high",
      "what_went_wrong": "Explain the mismatch between the selected and correct answers.",
      "possible_reason": "A plausible misconception, clearly marked as a hypothesis.",
      "reason_confidence": "low",
      "simple_explanation": "A beginner-friendly explanation.",
      "detailed_explanation": "A technically accurate explanation.",
      "important_rules": ["Rule 1"],
      "example": "A relevant example.",
      "topics_to_revisit": ["Topic 1"],
      "how_to_apply_correctly": ["Step 1"],
      "practice_recommendation": "A targeted practice task.",
      "verification_question": "A new question to check understanding."
    }}
  ]
}}

Requirements:
- Return one entry for each supplied incorrect question.
- Base explanations on the question, selected answer and correct answer.
- Do not claim to know why the student chose an answer.
- Treat reasons for mistakes as hypotheses, not facts.
- Do not invent the student's thought process.
- Explain the concept accurately and clearly.
- Set priority based on the educational importance and evidence available.
- Use the supplied question ID and question number.
- If a field cannot be established, use an empty string or an empty array.
- Return only complete, valid JSON. Do not use Markdown.

Incorrect questions:
{json.dumps(batch, ensure_ascii=False, default=str)}
"""

            batch_result = invoke_json(prompt)

            if not isinstance(batch_result, dict):
                raise ValueError(
                    "The knowledge-gap response must be a JSON object."
                )

            gaps = batch_result.get("knowledge_gaps", [])

            if isinstance(gaps, list):
                knowledge_gaps.extend(gaps)

    base_report["knowledge_gaps"] = knowledge_gaps

    # ---------------------------------------------------------
    # 9. Generate an overall report using compact evidence
    # ---------------------------------------------------------
    compact_attempts = []

    for quiz in quiz_results:
        compact_attempts.append({
            "quiz_id": quiz["quiz_id"],
            "attended": quiz["attended"],
            "attempt_count": quiz["attempt_count"],
            "attempts": [
                {
                    "percentage": attempt["percentage"],
                    "total_questions": attempt["total_questions"],
                    "correct_count": attempt["correct_count"],
                    "incorrect_count": attempt["incorrect_count"]
                }
                for attempt in quiz["attempts"]
            ]
        })

    overall_evidence = {
        "course_id": course_id,
        "module_id": module_id,
        "student_id": student_id,
        "total_quizzes": len(quiz_ids),
        "attempted_quizzes": base_report["attempted_quizzes"],
        "unattempted_quizzes": base_report["unattempted_quizzes"],
        "total_attempts": base_report["total_attempts"],
        "total_questions": total_questions,
        "total_correct": total_correct,
        "total_incorrect": base_report["total_incorrect"],
        "average_percentage": base_report["average_percentage"],
        "quiz_results": compact_attempts,
        "difficulty_analysis": difficulty_stats,
        "bloom_analysis": bloom_stats,
        "knowledge_gaps": knowledge_gaps
    }

    overall_prompt = f"""
You are an educational performance analyst.

Generate a student module performance report based only on the
supplied assessment evidence.

Return one valid JSON object with this exact structure:

{{
  "overall_summary": "Detailed performance summary.",
  "strengths": [
    {{
      "area": "Area",
      "evidence": "Supporting assessment evidence"
    }}
  ],
  "recommendations": [
    {{
      "priority": "high",
      "recommendation": "Actionable recommendation",
      "reason": "Why this action is relevant"
    }}
  ],
  "revision_roadmap": [
    {{
      "stage": 1,
      "topic": "Topic to revise",
      "activities": ["Activity 1"],
      "success_criteria": "How to measure understanding"
    }}
  ],
  "data_limitations": [
    "Limitation of the available assessment evidence"
  ]
}}

Instructions:
- Distinguish observed results from possible explanations.
- Do not infer mastery from an unattempted quiz.
- Do not invent strengths unsupported by correct answers or results.
- Use the knowledge gaps to create targeted recommendations.
- Include topics that should be learned or revisited.
- Give a practical revision roadmap.
- Mention insufficient evidence where appropriate.
- Do not repeat every question in the report.
- Keep the response concise enough to ensure complete JSON.
- Return only valid JSON, with no Markdown or additional text.

Assessment evidence:
{json.dumps(overall_evidence, ensure_ascii=False, default=str)}
"""

    overall = invoke_json(overall_prompt)

    if not isinstance(overall, dict):
        raise ValueError("The overall report must be a JSON object.")

    base_report["overall_summary"] = overall.get(
        "overall_summary", ""
    )
    base_report["strengths"] = overall.get("strengths", [])
    base_report["recommendations"] = overall.get(
        "recommendations", []
    )
    base_report["revision_roadmap"] = overall.get(
        "revision_roadmap", []
    )
    base_report["data_limitations"] = overall.get(
        "data_limitations", []
    )

    return base_report

In [71]:
report = summarize_student_module_performance(
    db_config=db_config,
    course_id=1,
    module_id=8,
    student_id="ITDA1282",
    llm=llm,
    batch_size=5,
    max_retries=2
)

print(json.dumps(report, indent=2, ensure_ascii=False))

{
  "course_id": 1,
  "module_id": 8,
  "student_id": "ITDA1282",
  "total_quizzes": 2,
  "attempted_quizzes": 1,
  "unattempted_quizzes": [
    9
  ],
  "total_attempts": 1,
  "total_questions": 10,
  "total_correct": 7,
  "total_incorrect": 3,
  "average_percentage": 70.0,
  "quiz_results": [
    {
      "quiz_id": 9,
      "attended": false,
      "attempt_count": 0,
      "attempts": [],
      "message": "The student has not attempted this quiz."
    },
    {
      "quiz_id": 4,
      "attended": true,
      "attempt_count": 1,
      "attempts": [
        {
          "attempt_id": 45,
          "score": 7.0,
          "total_questions": 10,
          "correct_count": 7,
          "incorrect_count": 3,
          "percentage": 70.0,
          "status": "submitted",
          "submitted_at": "2026-09-07T18:32:58",
          "duration_seconds": 134,
          "wrong_answers": [
            {
              "attempt_id": 45,
              "quiz_id": 4,
              "question_id": 31,
  